##Old frequency Branch on less compressed dataset version

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split , Dataset
import timm
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
GPU: NVIDIA L4


In [ ]:
import shutil
import os

if os.path.exists('/content/data_combined_c23_8k'):
    shutil.rmtree('/content/data_combined_c23_8k')

shutil.copytree('/content/drive/MyDrive/data_combined_c23_8k', '/content/data_combined_c23_8k')

print("Real frames:", len(os.listdir('/content/data_combined_c23_8k/real')))
print("Fake frames:", len(os.listdir('/content/data_combined_c23_8k/fake')))

Real frames: 8000
Fake frames: 8000


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print(os.path.isdir('/content/drive/MyDrive/data_combined_c23_8k'))

Mounted at /content/drive
True


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = self.subset.dataset.loader(path)
        return self.transform(image), label


full_dataset = ImageFolder('/content/data_combined_c23_8k', transform=None)
print("Classes:", full_dataset.classes)
print("Total:", len(full_dataset))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_sub, val_sub, test_sub = random_split(full_dataset, [train_size, val_size, test_size])

train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Classes: ['fake', 'real']
Total: 16000
Train: 11200, Val: 2400, Test: 2400


In [ ]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        fft = torch.fft.fft2(gray)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)) / \
                    (magnitude.std(dim=(-2, -1), keepdim=True) + 1e-6)
        out = self.features(magnitude)
        out = self.classifier(out)
        return out

class FrequencyOnlyDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_branch = FrequencyBranch()
        self.classifier = nn.Linear(128, 2)
    def forward(self, x):
        features = self.freq_branch(x)
        return self.classifier(features)

model = FrequencyOnlyDetector().cuda()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable parameters: 154,978


In [ ]:
num_epochs = 15

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

def val_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

print("Training functions ready")

Training functions ready


In [ ]:
best_val_acc = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc); val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_freqonlyc23.pth')
        print("  -> Best model saved")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

Epoch 1/15 | Train Loss: 0.6958 | Train Acc: 49.96% | Val Loss: 0.6911 | Val Acc: 53.83%
  -> Best model saved
Epoch 2/15 | Train Loss: 0.6937 | Train Acc: 51.12% | Val Loss: 0.6913 | Val Acc: 52.38%
Epoch 3/15 | Train Loss: 0.6933 | Train Acc: 50.85% | Val Loss: 0.6948 | Val Acc: 49.17%
Epoch 4/15 | Train Loss: 0.6924 | Train Acc: 51.35% | Val Loss: 0.6895 | Val Acc: 53.75%
Epoch 5/15 | Train Loss: 0.6931 | Train Acc: 50.96% | Val Loss: 0.6908 | Val Acc: 52.83%
Epoch 6/15 | Train Loss: 0.6928 | Train Acc: 51.12% | Val Loss: 0.6922 | Val Acc: 49.42%
Epoch 7/15 | Train Loss: 0.6920 | Train Acc: 51.98% | Val Loss: 0.6900 | Val Acc: 53.12%
Epoch 8/15 | Train Loss: 0.6920 | Train Acc: 51.81% | Val Loss: 0.6897 | Val Acc: 54.21%
  -> Best model saved
Epoch 9/15 | Train Loss: 0.6917 | Train Acc: 51.75% | Val Loss: 0.6907 | Val Acc: 52.29%
Epoch 10/15 | Train Loss: 0.6919 | Train Acc: 51.54% | Val Loss: 0.6899 | Val Acc: 53.42%
Epoch 11/15 | Train Loss: 0.6917 | Train Acc: 51.95% | Val Loss: 